# Compare ridge vs L1 -- mean(beta^2)/variance

Side-by-side (NOT over-plotted) `log(mean(beta^2)/variance)` panels for the same target, one
per fit. Reads two cached params subdirs, so both fits must already be computed+cached for
the dataset (dispatch each with the matching config `method`/`penalty`).

In [ ]:
import sys
from pathlib import Path
import json

_root = next(p for p in Path.cwd().resolve().parents if (p / '.git').exists() or (p / 'setup.py').exists())
for _p in (str(_root), str(_root / 'src')):
    if _p not in sys.path:
        sys.path.insert(0, _p)

import matplotlib.pyplot as plt
from metab_processing.LinearRegression import signature_helpers as sh, signature_batch as sb
from metab_processing.LinearRegression.signature_configs import CONFIGS

cfg = CONFIGS['breast']          # <- dataset to compare on
membership = json.load(open(Path(cfg.cache_dir) / '_membership.json'))

# The two fits to compare (each must be a computed params subdir).
L1    = dict(method='l1', penalty=0.005, l1_ratio=0.5)
RIDGE = dict(method='ridge', penalty=800)

TARGETS = sb.all_targets(cfg)
specs = sb.runspecs(cfg)
print(cfg.name, '| targets:', len(TARGETS), '| run labels:', [s['label'] for s in specs])

## `compare(sig, label)` -- two panels (L1 | ridge), mean(beta^2)/variance only

In [ ]:
def compare(sig, label=None, figsize=(20, 7)):
    """Side-by-side log(mean(beta^2)/variance) for `sig` at one run `label` (default: the
    first run-spec): left = L1, right = ridge. Separate axes -> no over-plotting."""
    label = label or specs[0]['label']
    hl = membership.get(sig, {}).get('present', [])
    fig, axes = plt.subplots(1, 2, figsize=figsize)
    for ax, params in zip(axes, (L1, RIDGE)):
        tag = f"{params['method']} p{params['penalty']}"
        res = sb.load_res(cfg, sig, label, **params)
        if res is None:
            ax.set_title(f'{sig} | {tag}\n(not computed)'); ax.axis('off'); continue
        sh.filter_and_plot_meansq_variance_ratio(res, ax=ax, highlight_genes=hl)
        ax.set_title(f'{sig} | {label} | {tag}', fontsize=9)
    fig.tight_layout()
    return fig

## Example -- one target

In [ ]:
compare('HALLMARK_INTERFERON_GAMMA_RESPONSE')

## All targets (first run-spec each)

In [ ]:
for sig in TARGETS:
    compare(sig)